In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

base = "/content/drive/MyDrive/Nizam/dataset_tokoku/"

pelanggan = pd.read_csv(base + "pelanggan.csv", sep=";", dtype=str)
produk = pd.read_csv(base + "produk.csv", sep=";", dtype=str)
pesanan = pd.read_csv(base + "pesanan.csv", sep=";", dtype=str)
detail = pd.read_csv(base + "detail_pesanan.csv", sep=";", dtype=str)

print(pelanggan.shape, produk.shape, pesanan.shape, detail.shape)

Mounted at /content/drive
(158, 6) (31, 5) (1515, 5) (3208, 6)


In [2]:
print(pesanan.duplicated().sum())          # baris duplikat persis
print(pesanan["status"].value_counts(dropna=False))
print(pelanggan["kota"].value_counts())

15
status
Selesai       672
Dibatalkan    227
Dikirim       136
selesai       127
SELESAI       122
Diproses      120
NaN           111
Name: count, dtype: int64
kota
Makassar      24
Semarang      21
Surabaya      17
Bandung       16
Bekasi        15
Tangerang     14
Denpasar      13
Medan         11
Jakarta       11
Yogyakarta    10
bandung        2
SURABAYA       1
JKT            1
Jakarta        1
Yogya          1
Name: count, dtype: int64


In [3]:
import numpy as np
import pandas as pd

# ---------- helper ----------
def parse_tgl(s):
    """Coba format yyyy-mm-dd dulu, lalu dd/mm/yyyy."""
    s = s.str.strip()
    t = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    return t.fillna(pd.to_datetime(s, format="%d/%m/%Y", errors="coerce"))

def norm_hp(x):
    if pd.isna(x):
        return np.nan
    x = x.strip()
    if x == "":
        return np.nan
    if x.startswith("+62"):
        x = "0" + x[3:]
    return x.replace("-", "").replace(" ", "")

KOTA = {"JAKARTA": "Jakarta", "JKT": "Jakarta", "BANDUNG": "Bandung", "BDG": "Bandung",
        "SURABAYA": "Surabaya", "SBY": "Surabaya", "YOGYAKARTA": "Yogyakarta", "YOGYA": "Yogyakarta"}
KATEGORI = {"ELEKTRONIK": "Elektronik", "FASHION": "Fashion", "RUMAH TANGGA": "Rumah Tangga",
            "OLAHRAGA": "Olahraga", "BUKU": "Buku"}
STATUS = {"SELESAI": "Selesai", "DIBATALKAN": "Dibatalkan", "DIKIRIM": "Dikirim", "DIPROSES": "Diproses"}

# ---------- 1. PRODUK ----------
produk_bersih = produk.copy()
produk_bersih["id_produk"] = produk_bersih["id_produk"].str.strip()
produk_bersih["nama_produk"] = produk_bersih["nama_produk"].str.strip()
produk_bersih["kategori"] = (produk["kategori"].str.strip().str.upper()
                             .map(KATEGORI).fillna(produk["kategori"].str.strip()))
produk_bersih["harga_satuan"] = pd.to_numeric(produk["harga_satuan"], errors="coerce")
produk_bersih["stok"] = pd.to_numeric(produk["stok"], errors="coerce")

# ---------- 2. PELANGGAN ----------
pelanggan_norm = pelanggan.copy()
pelanggan_norm["id_pelanggan"] = pelanggan["id_pelanggan"].str.strip()
pelanggan_norm["nama"] = pelanggan["nama"].str.strip()
pelanggan_norm["email"] = pelanggan["email"].str.strip().str.lower().replace("", np.nan)
pelanggan_norm["no_hp"] = pelanggan["no_hp"].apply(norm_hp)
pelanggan_norm["kota"] = pelanggan["kota"].str.strip().apply(
    lambda k: KOTA.get(k.upper(), k) if pd.notna(k) else k)
pelanggan_norm["tanggal_daftar"] = parse_tgl(pelanggan["tanggal_daftar"])

# deteksi duplikat: sama email ATAU sama no_hp; id yang lebih kecil dianggap asli
mapping = {}                      # id duplikat -> id utama
first_email, first_hp = {}, {}
for _, r in pelanggan_norm.sort_values("id_pelanggan").iterrows():
    asal = None
    if pd.notna(r["email"]) and r["email"] in first_email:
        asal = first_email[r["email"]]
    elif pd.notna(r["no_hp"]) and r["no_hp"] in first_hp:
        asal = first_hp[r["no_hp"]]
    if asal:
        mapping[r["id_pelanggan"]] = asal
    else:
        if pd.notna(r["email"]):
            first_email[r["email"]] = r["id_pelanggan"]
        if pd.notna(r["no_hp"]):
            first_hp[r["no_hp"]] = r["id_pelanggan"]

pelanggan_bersih = (pelanggan_norm[~pelanggan_norm["id_pelanggan"].isin(mapping.keys())]
                    .reset_index(drop=True))

# ---------- 3. PESANAN ----------
pesanan_bersih = pesanan.drop_duplicates(subset="id_pesanan").reset_index(drop=True)
pesanan_bersih["id_pesanan"] = pesanan_bersih["id_pesanan"].str.strip()

ids = pesanan_bersih["id_pelanggan"].str.strip().replace("", np.nan)
pesanan_bersih["id_pelanggan"] = ids.replace(mapping)     # arahkan duplikat ke id utama
pesanan_bersih["pelanggan_valid"] = (pesanan_bersih["id_pelanggan"]
                                     .isin(pelanggan_bersih["id_pelanggan"]).astype(int))

pesanan_bersih["tanggal_pesanan"] = parse_tgl(pesanan_bersih["tanggal_pesanan"])
pesanan_bersih["status"] = (pesanan_bersih["status"].str.strip().str.upper().map(STATUS))

metode = pesanan_bersih["metode_bayar"].str.strip()
pesanan_bersih["metode_bayar"] = metode.where(metode.str.upper() != "QRIS", "QRIS").replace("", np.nan)

# ---------- 4. DETAIL PESANAN ----------
detail_bersih = detail.drop_duplicates(subset="id_detail").reset_index(drop=True).copy()
detail_bersih["id_detail"] = detail_bersih["id_detail"].str.strip()
detail_bersih["id_pesanan"] = detail_bersih["id_pesanan"].str.strip()
detail_bersih["id_produk"] = detail_bersih["id_produk"].str.strip()

jumlah = pd.to_numeric(detail_bersih["jumlah"], errors="coerce")
detail_bersih["jumlah"] = jumlah.where(jumlah != 0).abs()     # 0 -> NaN, negatif -> positif

harga_map = produk_bersih.set_index("id_produk")["harga_satuan"]
harga_asal = pd.to_numeric(detail_bersih["harga_satuan"], errors="coerce")
detail_bersih["harga_satuan"] = harga_asal.fillna(detail_bersih["id_produk"].map(harga_map))
detail_bersih["diskon_persen"] = pd.to_numeric(detail_bersih["diskon_persen"], errors="coerce").fillna(0)
detail_bersih["produk_valid"] = detail_bersih["id_produk"].isin(produk_bersih["id_produk"]).astype(int)
detail_bersih["total_bayar"] = (detail_bersih["jumlah"] * detail_bersih["harga_satuan"]
                                * (100 - detail_bersih["diskon_persen"]) / 100)

# ---------- 5. VALIDASI ----------
print("pelanggan :", len(pelanggan), "->", len(pelanggan_bersih))
print("produk    :", len(produk), "->", len(produk_bersih))
print("pesanan   :", len(pesanan), "->", len(pesanan_bersih))
print("detail    :", len(detail), "->", len(detail_bersih))
print("duplikat pelanggan digabung:", len(mapping))
print("pesanan tanpa pelanggan valid:", (pesanan_bersih["pelanggan_valid"] == 0).sum())
print("detail tanpa produk valid   :", (detail_bersih["produk_valid"] == 0).sum())
print("\nStatus:\n", pesanan_bersih["status"].value_counts(dropna=False))
print("\nKota:\n", pelanggan_bersih["kota"].value_counts())

pelanggan : 158 -> 150
produk    : 31 -> 31
pesanan   : 1515 -> 1500
detail    : 3208 -> 3198
duplikat pelanggan digabung: 8
pesanan tanpa pelanggan valid: 111
detail tanpa produk valid   : 31

Status:
 status
Selesai       913
Dibatalkan    225
Dikirim       134
Diproses      118
NaN           110
Name: count, dtype: int64

Kota:
 kota
Makassar      24
Semarang      21
Bandung       17
Surabaya      16
Bekasi        15
Tangerang     14
Denpasar      12
Yogyakarta    11
Medan         10
Jakarta       10
Name: count, dtype: int64


In [10]:
import pandas as pd

# Hanya pesanan valid dan berstatus Selesai
valid = pesanan_bersih[
    (pesanan_bersih["pelanggan_valid"] == 1)
    & (pesanan_bersih["status"] == "Selesai")
].copy()

acuan = pesanan_bersih["tanggal_pesanan"].max()   # tanggal pesanan terakhir di data
punya_pesanan = valid["id_pelanggan"].nunique()   # pembagi: pelanggan dengan pesanan Selesai

print("Pelanggan yang punya pesanan Selesai:", punya_pesanan)

Pelanggan yang punya pesanan Selesai: 148


In [18]:
#Pertanyaan 1: Pelanggan mana yang mulai berhenti berbelanja?
terakhir = (valid.groupby("id_pelanggan")["tanggal_pesanan"]
            .max().rename("pesanan_terakhir").reset_index())
terakhir["hari_sejak_terakhir"] = (acuan - terakhir["pesanan_terakhir"]).dt.days

berhenti = (terakhir[terakhir["hari_sejak_terakhir"] > 90]
            .merge(pelanggan_bersih[["id_pelanggan", "nama", "kota"]], on="id_pelanggan")
            .sort_values("hari_sejak_terakhir", ascending=False))

print(f"Pelanggan berhenti: {len(berhenti)} dari {punya_pesanan}")
print(berhenti.head(10))

Pelanggan berhenti: 58 dari 148
   id_pelanggan pesanan_terakhir  hari_sejak_terakhir              nama  \
14       PLG029       2023-02-20                  494  Dimas Firmansyah   
11       PLG026       2023-04-16                  439       Sari Kusuma   
46       PLG124       2023-04-19                  436     Laras Saputri   
35       PLG093       2023-05-07                  418      Dani Santoso   
25       PLG065       2023-06-12                  382      Budi Santoso   
13       PLG028       2023-07-03                  361      Raka Saputri   
21       PLG055       2023-07-06                  358      Putri Wijaya   
51       PLG135       2023-07-30                  334         Tia Putra   
44       PLG120       2023-09-14                  288       Ayu Nugroho   
20       PLG050       2023-09-15                  287       Hadi Kusuma   

        kota  
14  Semarang  
11  Makassar  
46  Denpasar  
35  Semarang  
25   Bandung  
13     Medan  
21  Makassar  
51  Makassar  
44   Ja

In [19]:
#Pertanyaan 2: Seberapa sering pelanggan kembali berbelanja?
urut = valid.sort_values(["id_pelanggan", "tanggal_pesanan"]).copy()
urut["selisih_hari"] = urut.groupby("id_pelanggan")["tanggal_pesanan"].diff().dt.days

jumlah_pesanan = valid.groupby("id_pelanggan").size()
print(f"Pelanggan dengan lebih dari satu pesanan: {(jumlah_pesanan > 1).sum()} dari {punya_pesanan}")

selisih = urut["selisih_hari"].dropna()
print("\nJarak antar pesanan (hari):")
print(selisih.describe())
print("Median:", selisih.median())

Pelanggan dengan lebih dari satu pesanan: 143 dari 148

Jarak antar pesanan (hari):
count    696.000000
mean      74.205460
std       69.820444
min        0.000000
25%       22.000000
50%       53.000000
75%      102.250000
max      433.000000
Name: selisih_hari, dtype: float64
Median: 53.0


In [20]:
#Pertanyaan 3: Produk apa yang sering dibeli bersamaan?
selesai_ids = pesanan_bersih.loc[pesanan_bersih["status"] == "Selesai", "id_pesanan"]

item = (detail_bersih[           (detail_bersih["produk_valid"] == 1)
            & (detail_bersih["id_pesanan"].isin(selesai_ids))]
        [["id_pesanan", "id_produk"]].drop_duplicates())

pasangan = item.merge(item, on="id_pesanan", suffixes=("_a", "_b"))
pasangan = pasangan[pasangan["id_produk_a"] < pasangan["id_produk_b"]]

pasangan_top = (pasangan.groupby(["id_produk_a", "id_produk_b"]).size()
                .reset_index(name="jumlah_pesanan")
                .sort_values("jumlah_pesanan", ascending=False)
                .head(10)
                .reset_index(drop=True))

nama = produk_bersih.set_index("id_produk")["nama_produk"]
pasangan_top["produk_a"] = pasangan_top["id_produk_a"].map(nama)
pasangan_top["produk_b"] = pasangan_top["id_produk_b"].map(nama)
print("Pesanan Selesai yang dipakai:", len(selesai_ids))
print(pasangan_top[["produk_a", "produk_b", "jumlah_pesanan"]])

Pesanan Selesai yang dipakai: 913
             produk_a               produk_b  jumlah_pesanan
0  Earphone Bluetooth          Keyboard Slim              10
1      Mouse Wireless        Gelas Kaca 6pcs               9
2      Mouse Wireless       Blender Portable               9
3  Earphone Bluetooth          Kemeja Flanel               9
4       Skipping Rope  Buku Keuangan Pribadi               8
5  Earphone Bluetooth          Skipping Rope               8
6  Earphone Bluetooth        Resistance Band               8
7       Keyboard Slim       Lampu LED Pintar               8
8    Lampu LED Pintar          Kemeja Flanel               8
9  Rak Sepatu Plastik            Matras Yoga               8


In [21]:
#Pertanyaan 4: Apakah diskon mendorong pembelian lebih banyak?
from scipy.stats import mannwhitneyu

selesai = pesanan_bersih.loc[pesanan_bersih["status"] == "Selesai", ["id_pesanan"]]

d = detail_bersih.merge(selesai, on="id_pesanan")
d = d[d["jumlah"].notna() & (d["produk_valid"] == 1)].copy()
d["ada_diskon"] = d["diskon_persen"] > 0

print(d.groupby("ada_diskon")["jumlah"].agg(["mean", "median", "count"]))
print("\nPer level diskon:")
print(d.groupby("diskon_persen")["jumlah"].agg(["mean", "count"]))

tanpa = d.loc[~d["ada_diskon"], "jumlah"]
dengan = d.loc[d["ada_diskon"], "jumlah"]
stat, p = mannwhitneyu(dengan, tanpa, alternative="two-sided")
print(f"\nMann-Whitney p-value: {p:.4f}")

                mean  median  count
ada_diskon                         
False       3.039496     3.0   1190
True        2.927581     3.0    649

Per level diskon:
                   mean  count
diskon_persen                 
0.0            3.039496   1190
5.0            2.900990    202
10.0           2.983051    236
15.0           2.890995    211

Mann-Whitney p-value: 0.1053


In [22]:
#Pertanyaan 5: Nilai pesanan yang masih tertahan
nilai_pesanan = detail_bersih.groupby("id_pesanan")["total_bayar"].sum()

tertahan = pesanan_bersih[pesanan_bersih["status"].isin(["Diproses", "Dikirim"])].copy()
tertahan["nilai"] = tertahan["id_pesanan"].map(nilai_pesanan).fillna(0)

acuan = pesanan_bersih["tanggal_pesanan"].max()
batas = acuan - pd.Timedelta(days=30)
lama = tertahan[tertahan["tanggal_pesanan"] < batas]

print("Total nilai tertahan: Rp", f"{tertahan['nilai'].sum():,.0f}")
print("\nPer status:")
print(tertahan.groupby("status")["nilai"].agg(["sum", "count"]))
print("\nLebih dari 30 hari:", len(lama), "pesanan, Rp", f"{lama['nilai'].sum():,.0f}")

Total nilai tertahan: Rp 804,397,050

Per status:
                  sum  count
status                      
Dikirim   436828700.0    134
Diproses  367568350.0    118

Lebih dari 30 hari: 242 pesanan, Rp 757,860,775
